# AMPS I · Week 10 — HW9: Temporal convergence rates
**Due:** before the Week 11 lecture · **Points:** 100 · **Submit:** this notebook (`.ipynb`) with all outputs

**Rules**
- Before submitting: *Runtime → Restart session and run all*. Notebooks that do not run end to end receive 0.
- An error inside a `%%fenicsx` cell does **not** stop *Run all*. Read every output before you submit.
- LLM assistants are encouraged. Fill in the **AI-usage log** at the end.
- Every number you report must come from a cell you ran. For every rate: **prediction first**, then measurement, then explanation.

| Question | Topic | Points |
|---|---|:---:|
| Q1 | Amplification factor, stability limit, $\lambda_{\max}$ in 1D, energy stability | 20 |
| Q2 | θ-method MMS: temporal rates, joint refinement, order reduction | 40 |
| Q3 | Explicit Euler: $\lambda_{\max}$, $\Delta t_{\rm crit}$, $h^2$ scaling, P1 vs P2 | 20 |
| Q4 | LLM debugging: a heat solver with four bugs | 20 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## Q1 — Theory (20 pts)

**(a) (6) Amplification factor.** For $M\dot U+KU=0$ and an eigenpair $K\varphi=\lambda M\varphi$, $\lambda>0$, show that one θ-step multiplies the
$\varphi$-component by $R(z)=\frac{1-(1-\theta)z}{1+\theta z}$, $z=\lambda\Delta t$. Prove: $|R(z)|\le1$ for all $z\ge0$ iff $\theta\ge\frac12$; for $\theta<\frac12$,
$|R(z)|\le1$ iff $z\le\frac{2}{1-2\theta}$. Give $R(\infty)$ for $\theta=\frac12$ and $\theta=1$ and say what it means for stiff components.

**(b) (6) $\lambda_{\max}$ in 1D.** P1 on $(0,1)$ with $N$ equal cells, $h=1/N$, homogeneous Dirichlet conditions:
$K=\frac1h\,\mathrm{tridiag}(-1,2,-1)$, $M=\frac h6\,\mathrm{tridiag}(1,4,1)$. Show that $\varphi_i=\sin(j\pi x_i)$ are eigenvectors with
$\lambda_j=\frac{6}{h^2}\,\frac{1-\cos(j\pi h)}{2+\cos(j\pi h)}$, hence $\lambda_{\max}\to12/h^2$. With the lumped mass $hI$: $\lambda_j=\frac{2}{h^2}(1-\cos j\pi h)$,
$\lambda_{\max}\to4/h^2$. State the explicit-Euler limits for $\Delta t$. Check both formulas for $N=16$ with NumPy (next cell, no `%%fenicsx`).

**(c) (8) Energy stability.** For $u_t=\Delta u$, $u=0$ on $\partial\Omega$, and the fully discrete scheme
$(u^{n+1}-u^n,v)+\Delta t\,(\nabla u^{n+\theta},\nabla v)=0$, $u^{n+\theta}=\theta u^{n+1}+(1-\theta)u^n$, show
(i) backward Euler: $\|u^{n+1}\|\le\|u^n\|$ (take $v=u^{n+1}$); (ii) Crank–Nicolson: $\|u^{n+1}\|^2-\|u^n\|^2=-2\Delta t\,\|\nabla u^{n+1/2}\|^2$.
Explain why (ii) does not prevent the oscillations of lecture §C2.

> *your answer to (a)–(c)*

In [ ]:
# TODO: Q1(b) check the 1D eigenvalue formulas for N = 16 with NumPy

## Q2 — θ-method with a manufactured solution (40 pts)

$u_t-\Delta u=f$ on $(0,1)^2$, $0<t\le1$, $\tilde u=\sin(\pi x)\sin(\pi y)\cos(2\pi t)+xy\sin(\pi t)$ — the boundary data depend on time. `mms_setup` (given)
builds $\tilde u(t_{n+1})$, $f(t_n)$ and $f(t_{n+1})$ from two time `Constant`s; $f=\partial_t\tilde u-\Delta\tilde u$ by `ufl.diff` with respect to `ufl.variable(t)`.

**(a) (10)** Write `mms_solve(N, k, nt, theta, T=1.0, fmode="theta")`: θ-method with the source $\theta f^{n+1}+(1-\theta)f^n$, boundary data at $t_{n+1}$,
the matrix assembled and factorised **once**, an integer step loop. Return the $L^2$ error at $T$, the number of steps and the final time. Test: P2, $N=32$,
backward Euler, 20 steps (`-np 2`).

**(b) (12) Temporal rates.** P2, $\Delta t=1/10,\dots,1/160$: backward Euler and Crank–Nicolson on $N=64$, and Crank–Nicolson on $N=32$. Predict, measure,
explain the difference between the two CN columns.

**(c) (10) Joint refinement.** P1, $N=8,16,32,64$: (i) BE with $\Delta t=h$; (ii) CN with $\Delta t=h$; (iii) BE with $\Delta t=4h^2$ (`nt = N*N // 4`). Predict the
$L^2$ rate of each from $O(h^{k+1}+\Delta t^p)$ and measure it.

**(d) (8) Order reduction.** Crank–Nicolson, P2, $N=64$, with the source evaluated only at $t_{n+1}$ (`fmode="new"`) and only at $t_n$ (`"old"`).
Rates? At $\Delta t=1/10$ one of them has a **smaller** error than the correct scheme — why is that no argument for it?

In [ ]:
%%fenicsx -np 2
# TODO: Q2(a) write mms_solve and test it, -np 2

from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

def mms_setup(N, k):
    """u_t - Lap u = f on the unit square, u = u_ex on the boundary, 0 < t <= 1.
    u_ex = sin(pi x) sin(pi y) cos(2 pi t) + x y sin(pi t). Two time Constants: t_old = t_n, t_new = t_{n+1}.
    Returns msh, V, t_old, t_new, u_ex(t_new), f(t_old), f(t_new)."""
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("Lagrange", k))
    x = ufl.SpatialCoordinate(msh)
    t_old, t_new = fem.Constant(msh, 0.0), fem.Constant(msh, 0.0)
    def exact(tc):
        tv = ufl.variable(tc)
        u = ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1]) * ufl.cos(2 * ufl.pi * tv) + x[0] * x[1] * ufl.sin(ufl.pi * tv)
        return u, ufl.diff(u, tv) - ufl.div(ufl.grad(u))          # u and f = u_t - Lap u
    u_new, f_new = exact(t_new)
    _, f_old = exact(t_old)
    return msh, V, t_old, t_new, u_new, f_old, f_new

> *your answer to (a)*

In [ ]:
%%fenicsx -np 2
# TODO: Q2(b) temporal rates, -np 2

from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

def mms_setup(N, k):
    """u_t - Lap u = f on the unit square, u = u_ex on the boundary, 0 < t <= 1.
    u_ex = sin(pi x) sin(pi y) cos(2 pi t) + x y sin(pi t). Two time Constants: t_old = t_n, t_new = t_{n+1}.
    Returns msh, V, t_old, t_new, u_ex(t_new), f(t_old), f(t_new)."""
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("Lagrange", k))
    x = ufl.SpatialCoordinate(msh)
    t_old, t_new = fem.Constant(msh, 0.0), fem.Constant(msh, 0.0)
    def exact(tc):
        tv = ufl.variable(tc)
        u = ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1]) * ufl.cos(2 * ufl.pi * tv) + x[0] * x[1] * ufl.sin(ufl.pi * tv)
        return u, ufl.diff(u, tv) - ufl.div(ufl.grad(u))          # u and f = u_t - Lap u
    u_new, f_new = exact(t_new)
    _, f_old = exact(t_old)
    return msh, V, t_old, t_new, u_new, f_old, f_new

> *your prediction, table and answer to (b)*

In [ ]:
%%fenicsx -np 2
# TODO: Q2(c) joint refinement, -np 2

from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

def mms_setup(N, k):
    """u_t - Lap u = f on the unit square, u = u_ex on the boundary, 0 < t <= 1.
    u_ex = sin(pi x) sin(pi y) cos(2 pi t) + x y sin(pi t). Two time Constants: t_old = t_n, t_new = t_{n+1}.
    Returns msh, V, t_old, t_new, u_ex(t_new), f(t_old), f(t_new)."""
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("Lagrange", k))
    x = ufl.SpatialCoordinate(msh)
    t_old, t_new = fem.Constant(msh, 0.0), fem.Constant(msh, 0.0)
    def exact(tc):
        tv = ufl.variable(tc)
        u = ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1]) * ufl.cos(2 * ufl.pi * tv) + x[0] * x[1] * ufl.sin(ufl.pi * tv)
        return u, ufl.diff(u, tv) - ufl.div(ufl.grad(u))          # u and f = u_t - Lap u
    u_new, f_new = exact(t_new)
    _, f_old = exact(t_old)
    return msh, V, t_old, t_new, u_new, f_old, f_new

> *your prediction, table and answer to (c)*

In [ ]:
%%fenicsx -np 2
# TODO: Q2(d) order reduction, -np 2

from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

def mms_setup(N, k):
    """u_t - Lap u = f on the unit square, u = u_ex on the boundary, 0 < t <= 1.
    u_ex = sin(pi x) sin(pi y) cos(2 pi t) + x y sin(pi t). Two time Constants: t_old = t_n, t_new = t_{n+1}.
    Returns msh, V, t_old, t_new, u_ex(t_new), f(t_old), f(t_new)."""
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("Lagrange", k))
    x = ufl.SpatialCoordinate(msh)
    t_old, t_new = fem.Constant(msh, 0.0), fem.Constant(msh, 0.0)
    def exact(tc):
        tv = ufl.variable(tc)
        u = ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1]) * ufl.cos(2 * ufl.pi * tv) + x[0] * x[1] * ufl.sin(ufl.pi * tv)
        return u, ufl.diff(u, tv) - ufl.div(ufl.grad(u))          # u and f = u_t - Lap u
    u_new, f_new = exact(t_new)
    _, f_old = exact(t_old)
    return msh, V, t_old, t_new, u_new, f_old, f_new

> *your answer to (d)*

## Q3 — Explicit Euler: the stability limit, measured (20 pts)

Unit square, homogeneous Dirichlet conditions, $f=0$. Use dense matrices on the interior DOFs in a **serial** cell (`-np 1`; `dense_MK` is given).

**(a) (8)** P1, $N=8,16,32$: $\lambda_{\max}$, $\lambda_{\max}h^2$ (with $h=1/N$), $\Delta t_{\rm crit}=2/\lambda_{\max}$. Compare $\lambda_{\max}h^2$ with the 1D value of Q1(b).
How does $\Delta t_{\rm crit}$ scale, and how many explicit steps would you need to reach $t=1$ on $N=64$?

**(b) (6)** P1, $N=16$, initial data $u_0=1$ on the interior nodes (incompatible with $u=0$ on the boundary): run explicit Euler with $\Delta t=0.9$ and
$1.1\,\Delta t_{\rm crit}$ to $t=0.2$ and report $\max|u_h|$ at $t=0.05,\,0.1,\,0.2$. The exact solution decays like $e^{-2\pi^2t}$ — which run is right?

**(c) (6)** P2 on $N=8$ and $N=16$: $\lambda_{\max}$ and $\Delta t_{\rm crit}$. Compare with P1 on the same mesh and with P1 at the **same number of DOFs**. What does
this say about explicit time stepping with higher-order elements?

In [ ]:
%%fenicsx -np 1
# TODO: Q3(a)-(c), -np 1

from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

import scipy.linalg as sla

def dense_MK(N, k=1):
    """Dense M and K on the interior DOFs of the unit square (serial)."""
    msh = mesh.create_unit_square(MPI.COMM_SELF, N, N)
    V = fem.functionspace(msh, ("Lagrange", k))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    M = fem.assemble_matrix(fem.form(u * v * ufl.dx)).to_dense()
    K = fem.assemble_matrix(fem.form(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx)).to_dense()
    I = np.setdiff1d(np.arange(M.shape[0]), boundary_dofs(msh, V))
    return M[np.ix_(I, I)], K[np.ix_(I, I)]

> *your answers to (a)–(c)*

## Q4 — LLM debugging: a heat solver with four bugs (20 pts)

An LLM was asked for "backward Euler for $u_t-\Delta u=f$ on the unit square with the manufactured solution $\tilde u=(1+x^2+2y^2)e^{-t}$, $\Delta t=0.1$,
$T=1$, and the $L^2$ error at $T$ for $N=8,16,32$". It produced the code below. It runs without an error message.

```python
from mpi4py import MPI
import numpy as np, ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

def run(N, dt=0.1, T=1.0):
    msh = mesh.create_unit_square(MPI.COMM_WORLD, N, N)
    V = fem.functionspace(msh, ("Lagrange", 1))
    x = ufl.SpatialCoordinate(msh)
    t = 0.0
    u_ex = (1 + x[0]**2 + 2 * x[1]**2) * ufl.exp(-t)
    f = -(1 + x[0]**2 + 2 * x[1]**2) * ufl.exp(-t) - 6 * ufl.exp(-t)
    u_n = fem.Function(V); u_n.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    g = fem.Function(V); g.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    msh.topology.create_connectivity(1, 2)
    bc = fem.dirichletbc(g, fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology)))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    a = u * v * ufl.dx + dt * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
    L = (u_n + dt * f) * v * ufl.dx
    problem = LinearProblem(a, L, bcs=[bc], petsc_options_prefix=f"heat{N}_",
                            petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
    while t < T:
        t += dt
        uh = problem.solve()
        u_n = uh
    return np.sqrt(MPI.COMM_WORLD.allreduce(fem.assemble_scalar(fem.form((uh - u_ex)**2 * ufl.dx)), op=MPI.SUM))

for N in (8, 16, 32):
    e = run(N)
    if MPI.COMM_WORLD.rank == 0:
        print(N, e)
```

**(a) (8)** Run it (next cell) and find the **four** bugs (plus one inefficiency). For each: the line, what is wrong, the symptom, and how you detected it.
**(b) (8)** Corrected version: integer step loop, data at $t_{n+1}$, the matrix factorised once. Report the error at $T=1$ for $N=8,16,32$ with $\Delta t=0.1$
and with $\Delta t=0.1/4^j$ ($\Delta t\sim h^2$), and the rates. **(c) (4)** Why does the error with $\Delta t=0.1$ stop decreasing as $N$ grows?

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np, ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

def run(N, dt=0.1, T=1.0):
    msh = mesh.create_unit_square(MPI.COMM_WORLD, N, N)
    V = fem.functionspace(msh, ("Lagrange", 1))
    x = ufl.SpatialCoordinate(msh)
    t = 0.0
    u_ex = (1 + x[0]**2 + 2 * x[1]**2) * ufl.exp(-t)
    f = -(1 + x[0]**2 + 2 * x[1]**2) * ufl.exp(-t) - 6 * ufl.exp(-t)
    u_n = fem.Function(V); u_n.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    g = fem.Function(V); g.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    msh.topology.create_connectivity(1, 2)
    bc = fem.dirichletbc(g, fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology)))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    a = u * v * ufl.dx + dt * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
    L = (u_n + dt * f) * v * ufl.dx
    problem = LinearProblem(a, L, bcs=[bc], petsc_options_prefix=f"heat{N}_",
                            petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
    while t < T:
        t += dt
        uh = problem.solve()
        u_n = uh
    return np.sqrt(MPI.COMM_WORLD.allreduce(fem.assemble_scalar(fem.form((uh - u_ex)**2 * ufl.dx)), op=MPI.SUM))

for N in (8, 16, 32):
    e = run(N)
    if MPI.COMM_WORLD.rank == 0:
        print(N, e)

In [ ]:
%%fenicsx -np 2
# TODO: Q4 diagnostics, -np 2

> *your table for (a)*

In [ ]:
%%fenicsx -np 2
# TODO: Q4(b) corrected version, -np 2

> *your answer to (c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …